In [1]:
import sys

import sys
from cells_simulation import *

import sys
sys.path.insert(0, "../../")

from RealTrace_paper_analysis.plotting_src.read_ggp_run import * # you might need to change these two line, such that you can use read ggp_run
from RealTrace_paper_analysis.plotting_src.naive_rates import * 

import numpy as np
import subprocess

/Users/bjoern/mother_machine/RealTrace_figure_notebooks/simulations/../../RealTrace_paper_analysis/plotting_src/header_settings.py:8: UserWarning: A NumPy version >=1.22.4 and <2.3.0 is required for this version of SciPy (detected version 1.21.0)
  import scipy.stats


header_setting.py: using old naming


In [2]:

# ========== Simulation parameters ========== #
save_dataset = True    # saves data set as csv
run_ggp = True         # runs the ggp code (located in "../bin/" relative to this nb)

dt = 0.01
dt_measument = 3 # in minutes

n_cells = 1000 # number of cells that will be simulated
discard_cells = 20
lanes = 1

log_length0 = np.log(2)
gfp0 = 24975.8


div_mode = "adder"
division_log_length = None   # for sizer: division, when log_length hits division_log_length
division_time = None          # for timer: division, when cell cycle time hits division_time
division_addition = log_length0       # for adder: divsion, when division_addition in log_length was added in cell cycle


data_set_name = "glucose_hi1"
# taken from glucose_hi1_f01234578910_b_prediction in data_dany_old_data_output_20221215
parameter_sets  ={"glucose_hi1": {  "mean_lambda": 0.0114137, 
                                    "gamma_lambda": 0.166994,
                                    "var_lambda": 4.9481e-06,
                                    "mean_q": 118.524,
                                    "gamma_q": 0.0595599,
                                    "var_q": 430.055,
                                    "beta": 0, #0.0032,
                                    "var_x": 0.000136701,
                                    "var_g": 12.255,
                                    "var_dx": 0.00468653,
                                    "var_dg": 1.53195}}

parameter_set = parameter_sets[data_set_name]
parameter_set

{'mean_lambda': 0.0114137,
 'gamma_lambda': 0.166994,
 'var_lambda': 4.9481e-06,
 'mean_q': 118.524,
 'gamma_q': 0.0595599,
 'var_q': 430.055,
 'beta': 0,
 'var_x': 0.000136701,
 'var_g': 12.255,
 'var_dx': 0.00468653,
 'var_dg': 1.53195}

In [3]:
# --------- SIMULATE --------- #
cells_simulated = simulate_cells(dt, n_cells, parameter_set, div_mode,
                                division_log_length, 
                                division_time, division_addition, 
                                log_length0=log_length0, gfp0=gfp0, 
                                tree=True,
                                discard_cells=discard_cells)

In [4]:
base_dir ='../../data_realtrace_paper/{:s}_simulation{:d}_{:s}_{:s}_bleaching_mutliple_runs'

if parameter_set["beta"]>0:
    bleachin_flag="with"
else:
    bleachin_flag="wo"
    
out_dir = mk_missing_dir(base_dir.format(data_set_name, n_cells, div_mode, bleachin_flag))
out_dir

'../../data_realtrace_paper/glucose_hi1_simulation1000_adder_wo_bleaching_mutliple_runs'

In [5]:
# ----------- SAVE ----------- #
dt_measument = 3 # in minutes

dataset = build_data_set_scale_gfp_noise(cells_simulated, 
                                                parameter_set['var_x'], 
                                                parameter_set['var_g'], 
                                                dt_measument,
                                                atol=dt/10)

In [6]:

dataset["length_noise"] = np.exp(dataset["log_length_noise"])
directory, filename = get_next_file_name(out_dir)

write_csv_config(os.path.join(directory, "csv_config.txt"))

write_param_file(os.path.join(directory, "parameters_sim.txt"), parameter_set) 

dataset.to_csv(filename)



None


In [7]:
def add_noise_in_log(x, scale):
    return x * np.random.lognormal(0, sigma=scale)
    
def write_param_file_var_noise(filename, 
                               parameters, 
                               rel_step, 
                               mean_lambda_init,
                               mean_q_init):
    with open(filename, "w") as fin:
        
        parameters_init = {}
        n_level2 = 1
        fin.write("# Generated config file for simulated data with noise\n")
        
        range_lim=20
        
        mean_lambda =  mean_lambda_init
        fin.write("{:s} = {:.2E}, {:.2E}, {:.2E}, {:.2E}\n".format("mean_lambda", 
                                                   mean_lambda,
                                                   mean_lambda*0.001,
                                                   mean_lambda/10,
                                                   mean_lambda*10))
        
        gamma_lambda = add_noise_in_log(parameters["gamma_lambda"], n_level2)
        fin.write("{:s} = {:.2E}, {:.2E}, {:.2E}, {:.2E}\n".format("gamma_lambda", 
                                                   gamma_lambda,
                                                   gamma_lambda*rel_step,
                                                   gamma_lambda/range_lim,
                                                   gamma_lambda*range_lim))
        
        var_lambda = add_noise_in_log(parameters["var_lambda"], n_level2)
        fin.write("{:s} = {:.2E}, {:.2E}, {:.2E}, {:.2E}\n".format("var_lambda", 
                                                   var_lambda,
                                                   var_lambda*rel_step,
                                                   var_lambda/range_lim,
                                                   var_lambda*range_lim))
        
        mean_q = mean_q_init
        fin.write("{:s} = {:.2E}, {:.2E}, {:.2E}, {:.2E}\n".format("mean_q", 
                                                   mean_q,
                                                   mean_q*0.001,
                                                   mean_q/10,
                                                   mean_q*10))
        
        gamma_q = add_noise_in_log(parameters["gamma_q"], n_level2)
        fin.write("{:s} = {:.2E}, {:.2E}, {:.2E}, {:.2E}\n".format("gamma_q", 
                                                   gamma_q,
                                                   gamma_q*rel_step,
                                                   gamma_q/range_lim,
                                                   gamma_q*range_lim))
        
        fin.write("{:s} = {:.2E}\n".format("beta", parameters["beta"]))
        
        var_q = add_noise_in_log(parameters["var_q"], n_level2)
        fin.write("{:s} = {:.2E}, {:.2E}, {:.2E}, {:.2E}\n".format("var_q", 
                                                   var_q,
                                                   var_q*rel_step,
                                                   var_q/range_lim,
                                                   var_q*range_lim))
        
        var_x = add_noise_in_log(parameters["var_x"], n_level2)
        fin.write("{:s} = {:.2E}, {:.2E}, {:.2E}, {:.2E}\n".format("var_x", 
                                                   var_x,
                                                   var_x*rel_step,
                                                   var_x/range_lim,
                                                   var_x*range_lim))
        
        var_g = add_noise_in_log(parameters["var_g"], n_level2)
        fin.write("{:s} = {:.2E}, {:.2E}, {:.2E}, {:.2E}\n".format("var_g", 
                                                   var_g,
                                                   var_g*rel_step,
                                                   var_g/range_lim,
                                                   var_g*range_lim))
        
        var_dx = add_noise_in_log(parameters["var_dx"], n_level2)
        fin.write("{:s} = {:.2E}, {:.2E}, {:.2E}, {:.2E}\n".format("var_dx", 
                                                   var_dx,
                                                   var_dx*rel_step,
                                                   var_dx/range_lim,
                                                   var_dx*range_lim))
        
        var_dg = add_noise_in_log(parameters["var_dg"], n_level2)
        fin.write("{:s} = {:.2E}, {:.2E}, {:.2E}, {:.2E}\n".format("var_dg", 
                                                   var_dg,
                                                   var_dg*rel_step,
                                                   var_dg/range_lim,
                                                   var_dg*range_lim))
        


    return filename

In [8]:
np.random.seed(0)
for i in range(5):

    
    mean_lambda_init = np.mean(np.concatenate([cell.naive_rate for cell in naive_rate_for_all_cells(df2cells(dataset), 1, naive_growth_rate_skip_div)]))
    
    mean_q_init = np.mean(np.concatenate([cell.naive_rate for cell in naive_rate_for_all_cells(df2cells(dataset), 1, naive_production_rate_skip_div,beta=parameter_set['beta'])]))
    
    param_file = write_param_file_var_noise(os.path.join(directory, "parameters_noise{:d}.txt").format(i), 
                                            parameter_set, 
                                            0.1, 
                                            mean_lambda_init,
                                            mean_q_init)

    # param_file = write_param_file_noise(os.path.join(directory, "parameters_noise{:d}.txt").format(i), 
    #                                                 parameter_set, 
    #                                                 rel_step=0.1, 
    #                                                 log_noise=0.1, fixed=["beta"]) 
    
    # ----------- RUN ----------- #
    run_cmd = "RealTrace" + " -c " + os.path.join(directory, "csv_config.txt") + \
        " -b " + param_file + \
        " -t " + str(1e-40) +  " -i " + filename + " -l 0 " + '-m -p' +\
        " -o " + param_file[:-4]+ "_out"
    print("run: -> \n", run_cmd)
    s = subprocess.run(run_cmd, capture_output=True, shell=True)
    s

run: -> 
 RealTrace -c ../../data_realtrace_paper/glucose_hi1_simulation1000_adder_wo_bleaching_mutliple_runs/glucose_hi1_simulation1000_adder_wo_bleaching_mutliple_runs_0/csv_config.txt -b ../../data_realtrace_paper/glucose_hi1_simulation1000_adder_wo_bleaching_mutliple_runs/glucose_hi1_simulation1000_adder_wo_bleaching_mutliple_runs_0/parameters_noise0.txt -t 1e-40 -i ../../data_realtrace_paper/glucose_hi1_simulation1000_adder_wo_bleaching_mutliple_runs/glucose_hi1_simulation1000_adder_wo_bleaching_mutliple_runs_0/glucose_hi1_simulation1000_adder_wo_bleaching_mutliple_runs_0.csv -l 0 -m -p -o ../../data_realtrace_paper/glucose_hi1_simulation1000_adder_wo_bleaching_mutliple_runs/glucose_hi1_simulation1000_adder_wo_bleaching_mutliple_runs_0/parameters_noise0_out
run: -> 
 RealTrace -c ../../data_realtrace_paper/glucose_hi1_simulation1000_adder_wo_bleaching_mutliple_runs/glucose_hi1_simulation1000_adder_wo_bleaching_mutliple_runs_0/csv_config.txt -b ../../data_realtrace_paper/glucose_hi